In [1]:
import time
from datetime import datetime
from pyspark.sql import DataFrame, functions as F
  
SOURCE_TABLE = "silver.sales_obt"
GOLD_SCHEMA = "gold"
FACT_TABLE_NAME = "fact_orders"
 

StatementMeta(, 85c76e9d-fb68-456f-826e-0539e057deca, 3, Finished, Available, Finished, False)

In [2]:
GRAIN_KEYS = ["order_id", "order_item_id"]
 
NATURAL_KEYS = [
    "order_id", "order_item_id", "customer_id",
    "product_id", "store_id", "employee_id",
]
 
DESCRIPTIVE_COLUMNS = ["order_timestamp", "payment_method", "order_status"]
 
MEASURE_COLUMNS = ["quantity", "unit_price", "line_amount", "total_amount"]

AUDIT_COLUMNS = [
    "order_created_timestamp", "order_updated_timestamp",
    "order_processed_at", "obt_processed_at",
]
 
FACT_SOURCE_COLUMNS = NATURAL_KEYS + DESCRIPTIVE_COLUMNS + MEASURE_COLUMNS + AUDIT_COLUMNS
 

StatementMeta(, 85c76e9d-fb68-456f-826e-0539e057deca, 4, Finished, Available, Finished, False)

In [3]:
def create_fact_source(source_table: str = SOURCE_TABLE,
                        columns: list = None) -> DataFrame:
    columns = columns if columns is not None else FACT_SOURCE_COLUMNS
    columns_sql = ",\n    ".join(columns)
 
    sql = f"SELECT\n    {columns_sql}\nFROM {source_table}"
    print(f"[SOURCE SQL]\n{sql}\n")
 
    return spark.sql(sql)
 
 
def remove_duplicates(df: DataFrame, grain_keys: list = None) -> DataFrame:
    grain_keys = grain_keys if grain_keys is not None else GRAIN_KEYS
    before_count = df.count()
 
    deduped_df = df.dropDuplicates(grain_keys)
 
    after_count = deduped_df.count()
    removed = before_count - after_count
    print(f"[DEDUPE] rows before={before_count}, after={after_count}, removed={removed}")
 
    return deduped_df
 
 
def add_fact_metadata(df: DataFrame) -> DataFrame:
    return df.withColumn("fact_processed_at", F.current_timestamp())
 
 
def write_fact_table(df: DataFrame, target_table: str) -> None:
    print(f"[WRITE] {target_table}")
    (
        df.write
          .format("delta")
          .mode("overwrite")
          .option("overwriteSchema", "true")
          .saveAsTable(target_table)
    )

StatementMeta(, 85c76e9d-fb68-456f-826e-0539e057deca, 5, Finished, Available, Finished, False)

In [4]:
start_time = time.time()
start_ts = datetime.now()
 
full_target_table = f"{GOLD_SCHEMA}.{FACT_TABLE_NAME}"
 
fact_df = create_fact_source(SOURCE_TABLE, FACT_SOURCE_COLUMNS)
 
fact_df = remove_duplicates(fact_df, GRAIN_KEYS)
 
fact_df = add_fact_metadata(fact_df)
 
write_fact_table(fact_df, full_target_table)
 
end_time = time.time()
end_ts = datetime.now()

StatementMeta(, 85c76e9d-fb68-456f-826e-0539e057deca, 6, Finished, Available, Finished, False)

[SOURCE SQL]
SELECT
    order_id,
    order_item_id,
    customer_id,
    product_id,
    store_id,
    employee_id,
    order_timestamp,
    payment_method,
    order_status,
    quantity,
    unit_price,
    line_amount,
    total_amount,
    order_created_timestamp,
    order_updated_timestamp,
    order_processed_at,
    obt_processed_at
FROM silver.sales_obt

[DEDUPE] rows before=300513, after=30021, removed=270492
[WRITE] gold.fact_orders


In [5]:
final_df = spark.table(full_target_table)
row_count = final_df.count()
processing_seconds = round(end_time - start_time, 2)
 
print("=" * 80)
print(f"{full_target_table} — LOAD SUMMARY")
print("=" * 80)
print(f"Total row count      : {row_count}")
print(f"Processing started   : {start_ts.isoformat()}")
print(f"Processing finished  : {end_ts.isoformat()}")
print(f"Processing time (sec): {processing_seconds}")
print("-" * 80)
print("Schema:")
final_df.printSchema()
 
display(final_df.limit(20))

StatementMeta(, 85c76e9d-fb68-456f-826e-0539e057deca, 7, Finished, Available, Finished, False)

gold.fact_orders — LOAD SUMMARY
Total row count      : 30021
Processing started   : 2026-07-16T08:08:57.360611
Processing finished  : 2026-07-16T08:09:22.912704
Processing time (sec): 25.55
--------------------------------------------------------------------------------
Schema:
root
 |-- order_id: integer (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- store_id: integer (nullable = true)
 |-- employee_id: integer (nullable = true)
 |-- order_timestamp: timestamp (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: decimal(18,2) (nullable = true)
 |-- line_amount: decimal(18,2) (nullable = true)
 |-- total_amount: decimal(18,2) (nullable = true)
 |-- order_created_timestamp: timestamp (nullable = true)
 |-- order_updated_timestamp: timestamp (nullable = true)
 |-- order

SynapseWidget(Synapse.DataFrame, b93e8e65-f6a3-4414-a362-681bc88cad53)